Section 1 – Project Objective (Markdown)
# Feature Engineering

## Objective

The purpose of this notebook is to combine all cleaned datasets into a single master dataset and create meaningful business features that will support exploratory data analysis and the interactive dashboard.

The notebook includes:

- Loading cleaned datasets
- Validating relationships
- Merging datasets
- Creating business features
- Validating engineered features
- Exporting the final dataset

In [1]:
# Section 2 – Import Libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

plt.style.use("ggplot")

In [2]:
# Section 3 – Load Cleaned Datasets

customers = pd.read_csv("../dataset/cleaned/customers_cleaned.csv")

orders = pd.read_csv("../dataset/cleaned/orders_cleaned.csv")

order_items = pd.read_csv("../dataset/cleaned/order_items_cleaned.csv")

payments = pd.read_csv("../dataset/cleaned/payments_cleaned.csv")

products = pd.read_csv("../dataset/cleaned/products_cleaned.csv")

reviews = pd.read_csv("../dataset/cleaned/reviews_cleaned.csv")

sellers = pd.read_csv("../dataset/cleaned/sellers_cleaned.csv")

translation = pd.read_csv("../dataset/cleaned/translation_cleaned.csv")

geolocation = pd.read_csv("../dataset/cleaned/geolocation_cleaned.csv")

In [3]:
# Section 4 – Convert Date Columns

# Since CSV files don't preserve datetime types, convert them again.

order_dates = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_dates:
    orders[col] = pd.to_datetime(orders[col])

reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"]
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"]
)

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"]
)

In [4]:
# Section 5 – Relationship Validation

# Even though we checked this during cleaning, let's verify once more before merging.

# Orders → Customers

print(
    orders["customer_id"].isin(
        customers["customer_id"]
    ).all()
)

True


In [5]:
# Order Items → Orders

print(
    order_items["order_id"].isin(
        orders["order_id"]
    ).all()
)

True


In [6]:
# Order Items → Products

print(
    order_items["product_id"].isin(
        products["product_id"]
    ).all()
)

True


In [7]:
# Order Items → Sellers

print(
    order_items["seller_id"].isin(
        sellers["seller_id"]
    ).all()
)

True


In [8]:
# Payments → Orders

print(
    payments["order_id"].isin(
        orders["order_id"]
    ).all()
)

True


In [9]:
# Reviews → Orders

print(
    reviews["order_id"].isin(
        orders["order_id"]
    ).all()
)

True


In [10]:
# Section 6 – Merge Step 1
# Customers + Orders

master_df = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

master_df.shape

(99441, 12)

In [11]:
# Validate

master_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,Sao Paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,Barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,Vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,Sao Goncalo Do Amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,Santo Andre,SP


In [12]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
 8   customer_unique_id             99441 non-null  str           
 9   customer_zip_code_prefix       99441 non-null  int64         
 10  customer_city                  99441 non-null  str           
 11  customer_state            

In [13]:
# Section 7 – Merge Step 2

# Add Order Items.

master_df = master_df.merge(
    order_items,
    on="order_id",
    how="left"
)

master_df.shape

(113425, 18)

In [14]:
# Section 8 – Merge Step 3

# Add Products.

master_df = master_df.merge(
    products,
    on="product_id",
    how="left"
)

master_df.shape

(113425, 26)

In [15]:
# Section 9 – Merge Step 4

# Add Translation.

master_df = master_df.merge(
    translation,
    on="product_category_name",
    how="left"
)

master_df.shape

(113425, 27)

In [16]:
# Section 10 – Merge Step 5

# Add Sellers.

master_df = master_df.merge(
    sellers,
    on="seller_id",
    how="left"
)

master_df.shape

(113425, 30)

In [17]:
# Section 11 – Merge Step 6

# Add Payments.

master_df = master_df.merge(
    payments,
    on="order_id",
    how="left"
)

master_df.shape

(118434, 34)

In [18]:
# Section 12 – Merge Step 7

# Add Reviews.

master_df = master_df.merge(
    reviews,
    on="order_id",
    how="left"
)

master_df.shape

(119143, 42)

In [19]:
# Section 13 – Validate Master Dataset

master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 42 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       119143 non-null  str           
 1   customer_id                    119143 non-null  str           
 2   order_status                   119143 non-null  str           
 3   order_purchase_timestamp       119143 non-null  datetime64[us]
 4   order_approved_at              118966 non-null  datetime64[us]
 5   order_delivered_carrier_date   117057 non-null  datetime64[us]
 6   order_delivered_customer_date  115722 non-null  datetime64[us]
 7   order_estimated_delivery_date  119143 non-null  datetime64[us]
 8   customer_unique_id             119143 non-null  str           
 9   customer_zip_code_prefix       119143 non-null  int64         
 10  customer_city                  119143 non-null  str           
 11  customer_st

In [20]:
master_df.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_review_title,has_review_message
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,Sao Paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,Maua,SP,1.0,credit_card,1.0,18.12,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,False,True
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,Sao Paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,Maua,SP,3.0,voucher,1.0,2.00,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,False,True
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,Sao Paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,Maua,SP,2.0,voucher,1.0,18.59,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,False,True
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,Barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,Belo Horizonte,SP,1.0,boleto,1.0,141.46,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,True,True
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,Vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,Guariba,SP,1.0,credit_card,3.0,179.12,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18,2018-08-22 19:07:58,False,False


In [21]:
master_df.describe(include="all")


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_review_title,has_review_message
count,119143,119143,119143,119143,118966,117057,115722,119143,119143,119143.000000,119143,119143,118310.000000,118310,118310,118310,118310.000000,118310.000000,118310,116601.000000,116601.000000,116601.000000,118290.000000,118290.000000,118290.000000,118290.000000,116576,118310.000000,118310,118310,119140.000000,119140,119140.000000,119140.000000,118146,118146.000000,13989,50245,118146,118146,118146,118146
unique,99441,99441,8,NaN,NaN,NaN,NaN,NaN,96096,NaN,4119,27,NaN,32951,3095,NaN,NaN,NaN,74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71,NaN,611,23,NaN,5,NaN,NaN,98410,NaN,4527,36159,NaN,NaN,2,2
top,895ab968e7bb0d5659d16cd74cd1650c,270c23a11d024a44c896d1894b261a83,delivered,NaN,NaN,NaN,NaN,NaN,9a736b248f67d166d2fbb006bcb877c3,NaN,Sao Paulo,SP,NaN,aca2eb7d00ea1a7b8ebd4e68314663af,4a3ca9315b744ce9f8e9374361493884,NaN,NaN,NaN,cama_mesa_banho,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bed_bath_table,NaN,Sao Paulo,SP,NaN,credit_card,NaN,NaN,eef5dbca8d37dfce6db7d7b16dd0525e,NaN,Recomendo,Muito bom,NaN,NaN,False,False
freq,63,63,115723,NaN,NaN,NaN,NaN,NaN,75,NaN,18875,50265,NaN,536,2155,NaN,NaN,NaN,11988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11988,NaN,29293,84377,NaN,87776,NaN,NaN,63,NaN,494,259,NaN,NaN,104157,67901
mean,NaN,NaN,NaN,2017-12-29 18:36:13.115760,2017-12-30 04:49:18.425726,2018-01-03 08:24:34.395525,2018-01-12 20:55:38.199616,2018-01-22 15:21:10.241642,NaN,35033.451298,NaN,NaN,1.196543,NaN,NaN,2018-01-05 22:06:13.308807,120.646603,20.032387,NaN,48.767498,785.967822,2.205161,2112.250740,30.265145,16.619706,23.074799,NaN,24442.410413,NaN,NaN,1.094737,NaN,2.941246,172.735135,NaN,4.015582,NaN,NaN,2018-01-11 13:17:50.103093,2018-01-14 17:00:35.769302,NaN,NaN
min,NaN,NaN,NaN,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00,NaN,1003.000000,NaN,NaN,1.000000,NaN,NaN,2016-09-19 00:15:34,0.850000,0.000000,NaN,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000,NaN,1001.000000,NaN,NaN,1.000000,NaN,0.000000,0.000000,NaN,1.000000,NaN,NaN,2016-10-02 00:00:00,2016-10-07 18:32:28,NaN,NaN
25%,NaN,NaN,NaN,2017-09-10 20:15:46,2017-09-11 15:50:48.500000,2017-09-14 19:52:12,2017-09-22 21:54:31.250000,2017-10-02 00:00:00,NaN,11250.000000,NaN,NaN,1.000000,NaN,NaN,2017-09-18 14:30:33,39.900000,13.080000,NaN,42.000000,346.000000,1.000000,300.000000,18.000000,8.000000,15.000000,NaN,6429.000000,NaN,NaN,1.000000,NaN,1.000000,60.850000,NaN,4.000000,NaN,NaN,2017-09-22 00:00:00,2017-09-25 11:15:40.750000,NaN,NaN
50%,NaN,NaN,NaN,2018-01-17 11:59:12,2018-01-17 16:49:49,2018-01-23 17:03:08,2018-02-01 03:17:55,2018-02-14 00:00:00,NaN,24240.000000,NaN,NaN,1.000000,NaN,NaN,2018-01-25 04:11:15.500000,74.900000,16.280000,NaN,52.000000,600.000000,1.000000,700.000000,25.000000,13.000000,20.000000,NaN,13660.000000,NaN,NaN,1.000000,NaN,2.000000,108.160000,NaN,5.000000,NaN,NaN,2018-02-01 00:00:00,2018-02-03 12:04:23,NaN,NaN
75%,NaN,NaN,NaN,2018-05-03 13:18:30,2018-05-03 16:56:53,2018-05-07 14:57:00,2018-05-15 00:08:31.500000,2018-05-25 00:00:00,NaN,58475.000000,NaN,NaN,1.000000,NaN,NaN,2018-05-10 02:51:40.250000,134.900000,21.180000,NaN,57.000000,983.000000,3.000000,1800.000000,38.000000,20.000000,30.000000,NaN,27972.000000,NaN,NaN,1.000000,NaN,4.000000,189.240000,NaN,5.000000,NaN,NaN,2018-05

In [22]:
master_df.isnull().sum().sort_values(ascending=False).head(20)

review_comment_title             105154
review_comment_message            68898
order_delivered_customer_date      3421
product_category_name_english      2567
product_description_lenght         2542
product_name_lenght                2542
product_photos_qty                 2542
order_delivered_carrier_date       2086
review_id                           997
has_review_title                    997
review_creation_date                997
review_answer_timestamp             997
review_score                        997
has_review_message                  997
product_length_cm                   853
product_width_cm                    853
product_height_cm                   853
product_weight_g                    853
product_id                          833
seller_state                        833
dtype: int64

In [23]:
master_df.to_csv(
    "../exports/master_dataset.csv",
    index=False
)

Phase 3 - Section 14: Feature Engineering
Feature Categories
Date Features
Order Features
Customer Features
Seller Features
Product Features
Delivery Features
Payment Features
Review Features

# Part A - Date Features

These features make time-based analysis much easier.

In [24]:
# 1. Purchase Year

master_df["purchase_year"] = (
    master_df["order_purchase_timestamp"].dt.year
)

In [25]:
# 2. Purchase Month

master_df["purchase_month"] = (
    master_df["order_purchase_timestamp"].dt.month
)

In [26]:
# 3. Purchase Month Name

master_df["purchase_month_name"] = (
    master_df["order_purchase_timestamp"].dt.month_name()
)

In [27]:
# 4. Purchase Quarter

master_df["purchase_quarter"] = (
    master_df["order_purchase_timestamp"].dt.quarter
)

In [28]:
# 5. Purchase Weekday

master_df["purchase_weekday"] = (
    master_df["order_purchase_timestamp"].dt.day_name()
)

In [29]:
# 6. Purchase Hour

master_df["purchase_hour"] = (
    master_df["order_purchase_timestamp"].dt.hour
)

# Part B - Delivery Features

These measure the order fulfillment process.

In [30]:
# 7. Order Processing Time

# Time between purchase and approval.

master_df["order_processing_days"] = (
    master_df["order_approved_at"] -
    master_df["order_purchase_timestamp"]
).dt.days

In [31]:
# 8. Shipping Time

# Time from approval to carrier.

master_df["shipping_days"] = (
    master_df["order_delivered_carrier_date"] -
    master_df["order_approved_at"]
).dt.days

In [32]:
# 9. Delivery Time

# Time from carrier to customer.

master_df["delivery_days"] = (
    master_df["order_delivered_customer_date"] -
    master_df["order_delivered_carrier_date"]
).dt.days

In [33]:
# 10. Total Fulfillment Time

# Purchase → Delivery

master_df["total_delivery_days"] = (
    master_df["order_delivered_customer_date"] -
    master_df["order_purchase_timestamp"]
).dt.days

In [34]:
# 11. Delivery Delay

# Actual vs Estimated

master_df["delivery_delay_days"] = (
    master_df["order_delivered_customer_date"] -
    master_df["order_estimated_delivery_date"]
).dt.days

In [35]:
# 12. On-Time Delivery

master_df["on_time_delivery"] = np.where(
    master_df["delivery_delay_days"] <= 0,
    "Yes",
    "No"
)

# Part C - Revenue Features

In [36]:
# 13. Total Order Value
master_df["total_order_value"] = (
    master_df["price"] +
    master_df["freight_value"]
)

In [37]:
# 14. Freight Percentage

master_df["freight_percentage"] = (
    master_df["freight_value"] /
    master_df["total_order_value"]
) * 100

In [38]:
# 15. High Value Order

threshold = master_df["total_order_value"].median()

master_df["high_value_order"] = np.where(
    master_df["total_order_value"] >= threshold,
    "Yes",
    "No"
)

# Part D - Customer Features

In [39]:
# 16. Purchase Count

purchase_count = (
    master_df.groupby("customer_unique_id")["order_id"]
    .nunique()
)

master_df["customer_purchase_count"] = (
    master_df["customer_unique_id"]
    .map(purchase_count)
)

In [40]:
# 17. Repeat Customer

master_df["repeat_customer"] = np.where(
    master_df["customer_purchase_count"] > 1,
    "Yes",
    "No"
)

In [41]:
# 18. Customer Lifetime Value (CLV)

clv = (
    master_df.groupby("customer_unique_id")["total_order_value"]
    .sum()
)

master_df["customer_lifetime_value"] = (
    master_df["customer_unique_id"]
    .map(clv)
)

# Part E - Seller Features

In [42]:
# 19. Seller Revenue

seller_revenue = (
    master_df.groupby("seller_id")["total_order_value"]
    .sum()
)

master_df["seller_revenue"] = (
    master_df["seller_id"]
    .map(seller_revenue)
)

In [43]:
# 20. Seller Order Count

seller_orders = (
    master_df.groupby("seller_id")["order_id"]
    .nunique()
)

master_df["seller_order_count"] = (
    master_df["seller_id"]
    .map(seller_orders)
)

# Part F - Product Features

In [44]:
# 21. Average Product Price

avg_price = (
    master_df.groupby("product_id")["price"]
    .mean()
)

master_df["average_product_price"] = (
    master_df["product_id"]
    .map(avg_price)
)

In [45]:
# 22. Product Sales Count

product_sales = (
    master_df.groupby("product_id")["order_id"]
    .count()
)

master_df["product_sales_count"] = (
    master_df["product_id"]
    .map(product_sales)
)

# Part G - Review Features

In [46]:
# 23. Average Review Score

avg_review = (
    master_df.groupby("product_id")["review_score"]
    .mean()
)

master_df["average_review_score"] = (
    master_df["product_id"]
    .map(avg_review)
)

In [47]:
# 24. Positive Review

master_df["positive_review"] = np.where(
    master_df["review_score"] >= 4,
    "Yes",
    "No"
)

# Part H - Payment Features

In [48]:
# 25. Installment Payment

master_df["installment_payment"] = np.where(
    master_df["payment_installments"] > 1,
    "Yes",
    "No"
)

In [49]:
# Validate Features

# Check the new columns.

master_df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english', 'seller_zip_code_prefix',
       'seller_city', 'seller_state', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'review_id', 'review_score',
       'review_comment_title', 'review_comment_message',
       'review_creation_date', 'review_answer_timestamp', 'has_review_title',
       'has_review_message', 'pur

In [50]:
# View a sample.

master_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_review_title,has_review_message,purchase_year,purchase_month,purchase_month_name,purchase_quarter,purchase_weekday,purchase_hour,order_processing_days,shipping_days,delivery_days,total_delivery_days,delivery_delay_days,on_time_delivery,total_order_value,freight_percentage,high_value_order,customer_purchase_count,repeat_customer,customer_lifetime_value,seller_revenue,seller_order_count,average_product_price,product_sales_count,average_review_score,positive_review,installment_payment
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,Sao Paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,Maua,SP,1.0,credit_card,1.0,18.12,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,False,True,2017,10,October,4,Monday,10,0.0,2.0,6.0,8.0,-8.0,Yes,38.71,22.526479,No,2,Yes,160.24,3280.90,53.0,29.990000,6.0,4.00000,Yes,No
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,Sao Paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,Maua,SP,3.0,voucher,1.0,2.00,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,False,True,2017,10,October,4,Monday,10,0.0,2.0,6.0,8.0,-8.0,Yes,38.71,22.526479,No,2,Yes,160.24,3280.90,53.0,29.990000,6.0,4.00000,Yes,No
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,Sao Paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,Maua,SP,2.0,voucher,1.0,18.59,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,False,True,2017,10,October,4,Monday,10,0.0,2.0,6.0,8.0,-8.0,Yes,38.71,22.526479,No,2,Yes,160.24,3280.90,53.0,29.990000,6.0,4.00000,Yes,No
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,Barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,Belo Horizonte,SP,1.0,boleto,1.0,141.46,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,True,True,2018,7,July,3,Tuesday,20,1.0,0.0,12.0,13.0,-6.0,Yes,141.46,16.089354,Yes,1,No,141.46,15639.29,110.0,119.216038,106.0,4.40566,Yes,No
4,47770eb9100c2d0c44946d9cf07ec65d,

In [51]:
# Check missing values in engineered features.

master_df[
    [
        "customer_lifetime_value",
        "seller_revenue",
        "delivery_days",
        "delivery_delay_days"
    ]
].isnull().sum()

customer_lifetime_value       0
seller_revenue              833
delivery_days              3422
delivery_delay_days        3421
dtype: int64

In [52]:
# Save Feature Engineered Dataset

master_df.to_csv(
    "../exports/feature_engineered_dataset.csv",
    index=False
)